In [1]:
#Necessary packages to load data
import os
import xarray as xr
import numpy as np

In [2]:
base_dir =r"D:\ERA 5 Data\ERA5_Camden"

In [3]:
#Load in data
#For future use: add new variables using the same pattern
var_config = {
    "temp": {
        "folder": "Temp",
        "pattern": "era5_2mtemp__*.nc"
    },
    "u10": {
        "folder": "U Vector Wind",
        "pattern": "era5_uvector_*.nc"
    },
    "v10": {
        "folder": "V Vector Wind",
        "pattern": "era5_vvector_*.nc"
    },
    "sp": {
        "folder": "Surface Pressure",
        "pattern": "era5_sp__*.nc"
    },
    "dp": {
        "folder": "Dewpoint Temp",
        "pattern": "era5_2m_dew_temp__*.nc"
    },
    "k": {
        "folder": "K Index",
        "pattern": "era5_k_index_temp__*.nc"
    },
    "sol": {
        "folder": "Sol Rad",
        "pattern": "era5_surface_net_solar_radiation__*.nc"
    },
}

In [4]:
import glob

# Calculate Relative Humidity year-by-year

# Output folder
rh_folder = os.path.join(base_dir, "Relative Humidity")
os.makedirs(rh_folder, exist_ok=True)

# Years to process
years = range(2004, 2025)

for year in years:

    print(f"Processing {year}...")

    # Find the temperature and dew point files

    temp_pattern = os.path.join(
        base_dir,
        var_config["temp"]["folder"],
        f"era5_2mtemp__{year}.nc"
    )

    dp_pattern = os.path.join(
        base_dir,
        var_config["dp"]["folder"],
        f"era5_2m_dew_temp__{year}.nc"
    )

    # Check that both files exist
    if not os.path.exists(temp_pattern):
        print(f"  Temperature file not found: {temp_pattern}")
        continue

    if not os.path.exists(dp_pattern):
        print(f"  Dew point file not found: {dp_pattern}")
        continue

    # Load the year's data

    temp_ds = xr.open_dataset(temp_pattern)
    dp_ds = xr.open_dataset(dp_pattern)

    # Get the actual variable names
    temp_var = list(temp_ds.data_vars)[0]
    dp_var = list(dp_ds.data_vars)[0]

    print(f"  Temperature variable: {temp_var}")
    print(f"  Dew point variable: {dp_var}")

    # Convert Kelvin → Celsius
    T = temp_ds[temp_var] - 273.15
    Td = dp_ds[dp_var] - 273.15
    Pw = 6.11 * 10 ** (
        (7.5 * Td) / (237.3 + Td)
    )
    Pws = 6.11 * 10 ** (
        (7.5 * T) / (237.3 + T)
    )

    RH = (Pw / Pws) * 100

    # Keep RH physically between 0 and 100%
    RH = RH.clip(0, 100)

    # Give the variable a name
    RH.name = "relative_humidity"

    # Add metadata
    RH.attrs["long_name"] = "Relative Humidity"
    RH.attrs["units"] = "%"
    RH.attrs["description"] = (
        "Relative humidity calculated from 2-m air temperature "
        "and 2-m dew point temperature using vapor pressure equations."
    )

    # Save the year

    output_file = os.path.join(
        rh_folder,
        f"relative_humidity_{year}.nc"
    )

    RH.to_netcdf(output_file)

    # Close input files
    temp_ds.close()
    dp_ds.close()

    print(f"  Saved: {output_file}")

print("\nRelative humidity calculation complete!")

Processing 2004...
  Temperature variable: t2m
  Dew point variable: d2m
  Saved: D:\ERA 5 Data\ERA5_Camden\Relative Humidity\relative_humidity_2004.nc
Processing 2005...
  Temperature variable: t2m
  Dew point variable: d2m
  Saved: D:\ERA 5 Data\ERA5_Camden\Relative Humidity\relative_humidity_2005.nc
Processing 2006...
  Temperature variable: t2m
  Dew point variable: d2m
  Saved: D:\ERA 5 Data\ERA5_Camden\Relative Humidity\relative_humidity_2006.nc
Processing 2007...
  Temperature variable: t2m
  Dew point variable: d2m
  Saved: D:\ERA 5 Data\ERA5_Camden\Relative Humidity\relative_humidity_2007.nc
Processing 2008...
  Temperature variable: t2m
  Dew point variable: d2m
  Saved: D:\ERA 5 Data\ERA5_Camden\Relative Humidity\relative_humidity_2008.nc
Processing 2009...
  Temperature variable: t2m
  Dew point variable: d2m
  Saved: D:\ERA 5 Data\ERA5_Camden\Relative Humidity\relative_humidity_2009.nc
Processing 2010...
  Temperature variable: t2m
  Dew point variable: d2m
  Saved: D:\ERA

In [5]:
# Calculate Wind Speed year-by-year

# Output folder
ws_folder = os.path.join(base_dir, "Wind Speed")
os.makedirs(ws_folder, exist_ok=True)

# Years to process
years = range(2004, 2025)

for year in years:

    print(f"Processing {year}...")

    # --------------------------------------------------
    # Find the U and V vector files
    # --------------------------------------------------

    u_pattern = os.path.join(
        base_dir,
        var_config["u10"]["folder"],
        f"era5_uvector__{year}.nc"
    )

    v_pattern = os.path.join(
        base_dir,
        var_config["v10"]["folder"],
        f"era5_vvector__{year}.nc"
    )

    # Check that both files exist
    if not os.path.exists(u_pattern):
        print(f"  U vector file not found: {u_pattern}")
        continue

    if not os.path.exists(v_pattern):
        print(f"  V vector file not found: {v_pattern}")
        continue

    # --------------------------------------------------
    # Load the year's data
    # --------------------------------------------------

    u_ds = xr.open_dataset(u_pattern)
    v_ds = xr.open_dataset(v_pattern)

    # Get the actual variable names
    u_var = list(u_ds.data_vars)[0]
    v_var = list(v_ds.data_vars)[0]

    print(f"  U Vector variable: {u_var}")
    print(f"  V Vector variable: {v_var}")

    # --------------------------------------------------
    # Extract the actual data
    # --------------------------------------------------

    u = u_ds[u_var]
    v = v_ds[v_var]

    # --------------------------------------------------
    # Calculate wind speed
    #
    # Speed = sqrt(u² + v²)
    # --------------------------------------------------

    ws = np.sqrt(u**2 + v**2)

    # --------------------------------------------------
    # Give the variable a name
    # --------------------------------------------------

    ws.name = "wind_speed"

    # Add metadata
    ws.attrs["long_name"] = "10-m Wind Speed"
    ws.attrs["units"] = "m s-1"
    ws.attrs["description"] = (
        "Wind speed calculated from 10-m U and V wind "
        "components using the Pythagorean theorem."
    )

    # --------------------------------------------------
    # Save the year
    # --------------------------------------------------

    output_file = os.path.join(
        ws_folder,
        f"wind_speed_{year}.nc"
    )

    ws.to_netcdf(output_file)

    # Close input files
    u_ds.close()
    v_ds.close()

    print(f"  Saved: {output_file}")

print("\nWind Speed calculation complete!")

Processing 2004...
  U Vector variable: u10
  V Vector variable: v10
  Saved: D:\ERA 5 Data\ERA5_Camden\Wind Speed\wind_speed_2004.nc
Processing 2005...
  U Vector variable: u10
  V Vector variable: v10
  Saved: D:\ERA 5 Data\ERA5_Camden\Wind Speed\wind_speed_2005.nc
Processing 2006...
  U Vector variable: u10
  V Vector variable: v10
  Saved: D:\ERA 5 Data\ERA5_Camden\Wind Speed\wind_speed_2006.nc
Processing 2007...
  U Vector variable: u10
  V Vector variable: v10
  Saved: D:\ERA 5 Data\ERA5_Camden\Wind Speed\wind_speed_2007.nc
Processing 2008...
  U Vector variable: u10
  V Vector variable: v10
  Saved: D:\ERA 5 Data\ERA5_Camden\Wind Speed\wind_speed_2008.nc
Processing 2009...
  U Vector variable: u10
  V Vector variable: v10
  Saved: D:\ERA 5 Data\ERA5_Camden\Wind Speed\wind_speed_2009.nc
Processing 2010...
  U Vector variable: u10
  V Vector variable: v10
  Saved: D:\ERA 5 Data\ERA5_Camden\Wind Speed\wind_speed_2010.nc
Processing 2011...
  U Vector variable: u10
  V Vector variabl

https://www.wpc.ncep.noaa.gov/html/heatindex_equation.shtml

In [6]:
# Calculate Heat Index year-by-year

# Output folder
hi_folder = os.path.join(base_dir, "Heat Index")
os.makedirs(hi_folder, exist_ok=True)

# Years to process
years = range(2004, 2025)

for year in years:

    print(f"Processing {year}...")

    # --------------------------------------------------
    # Find the temperature and relative humidity files
    # --------------------------------------------------

    temp_pattern = os.path.join(
        base_dir,
        var_config["temp"]["folder"],
        f"era5_2mtemp__{year}.nc"
    )

    rh_pattern = os.path.join(
        base_dir,
        "Relative Humidity",
        f"relative_humidity_{year}.nc"
    )

    # Check that both files exist
    if not os.path.exists(temp_pattern):
        print(f"  Temperature file not found: {temp_pattern}")
        continue

    if not os.path.exists(rh_pattern):
        print(f"  Relative humidity file not found: {rh_pattern}")
        continue

    # --------------------------------------------------
    # Load the year's data
    # --------------------------------------------------

    temp_ds = xr.open_dataset(temp_pattern)
    rh_ds = xr.open_dataset(rh_pattern)

    # Get the actual variable names
    temp_var = list(temp_ds.data_vars)[0]
    rh_var = list(rh_ds.data_vars)[0]

    print(f"  Temperature variable: {temp_var}")
    print(f"  Relative humidity variable: {rh_var}")

    # --------------------------------------------------
    # Extract the actual data
    # --------------------------------------------------

    T_K = temp_ds[temp_var]
    RH = rh_ds[rh_var]

    # --------------------------------------------------
    # Convert temperature:
    # Kelvin → Celsius → Fahrenheit
    # --------------------------------------------------

    T = (T_K - 273.15) * 9/5 + 32

    # --------------------------------------------------
    # Heat Index equation
    #
    # HI = -42.379
    #      + 2.04901523*T
    #      + 10.14333127*RH
    #      - 0.22475541*T*RH
    #      - 0.00683783*T²
    #      - 0.05481717*RH²
    #      + 0.00122874*T²*RH
    #      + 0.00085282*T*RH²
    #      - 0.00000199*T²*RH²
    # --------------------------------------------------

    HI = (
        -42.379
        + 2.04901523 * T
        + 10.14333127 * RH
        - 0.22475541 * T * RH
        - 0.00683783 * T**2
        - 0.05481717 * RH**2
        + 0.00122874 * T**2 * RH
        + 0.00085282 * T * RH**2
        - 0.00000199 * T**2 * RH**2
    )

    # --------------------------------------------------
    # Give the variable a name
    # --------------------------------------------------

    HI.name = "heat_index"

    # Add metadata
    HI.attrs["long_name"] = "Heat Index"
    HI.attrs["units"] = "degF"
    HI.attrs["description"] = (
        "Heat Index calculated from 2-m air temperature "
        "and relative humidity using the Rothfusz regression equation."
    )

    # --------------------------------------------------
    # Save the year
    # --------------------------------------------------

    output_file = os.path.join(
        hi_folder,
        f"heat_index_{year}.nc"
    )

    HI.to_netcdf(output_file)

    # Close input files
    temp_ds.close()
    rh_ds.close()

    print(f"  Saved: {output_file}")

print("\nHeat Index calculation complete!")

Processing 2004...
  Temperature variable: t2m
  Relative humidity variable: relative_humidity
  Saved: D:\ERA 5 Data\ERA5_Camden\Heat Index\heat_index_2004.nc
Processing 2005...
  Temperature variable: t2m
  Relative humidity variable: relative_humidity
  Saved: D:\ERA 5 Data\ERA5_Camden\Heat Index\heat_index_2005.nc
Processing 2006...
  Temperature variable: t2m
  Relative humidity variable: relative_humidity
  Saved: D:\ERA 5 Data\ERA5_Camden\Heat Index\heat_index_2006.nc
Processing 2007...
  Temperature variable: t2m
  Relative humidity variable: relative_humidity
  Saved: D:\ERA 5 Data\ERA5_Camden\Heat Index\heat_index_2007.nc
Processing 2008...
  Temperature variable: t2m
  Relative humidity variable: relative_humidity
  Saved: D:\ERA 5 Data\ERA5_Camden\Heat Index\heat_index_2008.nc
Processing 2009...
  Temperature variable: t2m
  Relative humidity variable: relative_humidity
  Saved: D:\ERA 5 Data\ERA5_Camden\Heat Index\heat_index_2009.nc
Processing 2010...
  Temperature variabl

https://journals.ametsoc.org/view/journals/apme/50/11/jamc-d-11-0143.1.xml

In [7]:
# Calculate Wet-Bulb Temperature year-by-year
# Stull (2011) approximation

# Output folder
wb_folder = os.path.join(base_dir, "Wet Bulb Temp")
os.makedirs(wb_folder, exist_ok=True)

# Years to process
years = range(2004, 2025)

for year in years:

    print(f"Processing {year}...")

    # --------------------------------------------------
    # Find the temperature and relative humidity files
    # --------------------------------------------------

    temp_pattern = os.path.join(
        base_dir,
        var_config["temp"]["folder"],
        f"era5_2mtemp__{year}.nc"
    )

    rh_pattern = os.path.join(
        base_dir,
        "Relative Humidity",
        f"relative_humidity_{year}.nc"
    )

    # Check that both files exist
    if not os.path.exists(temp_pattern):
        print(f"  Temperature file not found: {temp_pattern}")
        continue

    if not os.path.exists(rh_pattern):
        print(f"  Relative humidity file not found: {rh_pattern}")
        continue

    # --------------------------------------------------
    # Load the year's data
    # --------------------------------------------------

    temp_ds = xr.open_dataset(temp_pattern)
    rh_ds = xr.open_dataset(rh_pattern)

    # Get the actual variable names
    temp_var = list(temp_ds.data_vars)[0]
    rh_var = list(rh_ds.data_vars)[0]

    print(f"  Temperature variable: {temp_var}")
    print(f"  Relative humidity variable: {rh_var}")

    # --------------------------------------------------
    # Extract the actual data
    # --------------------------------------------------

    # ERA5 temperature: Kelvin → Celsius
    T = temp_ds[temp_var] - 273.15

    # Relative humidity: %
    RH = rh_ds[rh_var]

    # --------------------------------------------------
    # Stull wet-bulb temperature formula
    # --------------------------------------------------

    WB = (
        T * np.arctan(
            0.151977 * np.sqrt(RH + 8.313659)
        )
        + np.arctan(T + RH)
        - np.arctan(RH - 1.676331)
        + 0.00391838 * RH**(3/2)
            * np.arctan(0.023101 * RH)
        - 4.686035
    )

    # --------------------------------------------------
    # Give the variable a name
    # --------------------------------------------------

    WB.name = "wet_bulb_temperature"

    # Add metadata
    WB.attrs["long_name"] = "Wet-Bulb Temperature"
    WB.attrs["units"] = "degC"
    WB.attrs["description"] = (
        "Wet-bulb temperature calculated using the "
        "Stull (2011) approximation from 2-m air temperature "
        "and relative humidity."
    )

    # --------------------------------------------------
    # Save the year
    # --------------------------------------------------

    output_file = os.path.join(
        wb_folder,
        f"wet_bulb_temperature_{year}.nc"
    )

    WB.to_netcdf(output_file)

    # Close input files
    temp_ds.close()
    rh_ds.close()

    print(f"  Saved: {output_file}")

print("\nWet-Bulb Temperature calculation complete!")

Processing 2004...
  Temperature variable: t2m
  Relative humidity variable: relative_humidity
  Saved: D:\ERA 5 Data\ERA5_Camden\Wet Bulb Temp\wet_bulb_temperature_2004.nc
Processing 2005...
  Temperature variable: t2m
  Relative humidity variable: relative_humidity
  Saved: D:\ERA 5 Data\ERA5_Camden\Wet Bulb Temp\wet_bulb_temperature_2005.nc
Processing 2006...
  Temperature variable: t2m
  Relative humidity variable: relative_humidity
  Saved: D:\ERA 5 Data\ERA5_Camden\Wet Bulb Temp\wet_bulb_temperature_2006.nc
Processing 2007...
  Temperature variable: t2m
  Relative humidity variable: relative_humidity
  Saved: D:\ERA 5 Data\ERA5_Camden\Wet Bulb Temp\wet_bulb_temperature_2007.nc
Processing 2008...
  Temperature variable: t2m
  Relative humidity variable: relative_humidity
  Saved: D:\ERA 5 Data\ERA5_Camden\Wet Bulb Temp\wet_bulb_temperature_2008.nc
Processing 2009...
  Temperature variable: t2m
  Relative humidity variable: relative_humidity
  Saved: D:\ERA 5 Data\ERA5_Camden\Wet B